# 02 · Draw the annotation sample
**Paper:** Step 2 · Fig. 2B

We select **1,000 impure images** for manual annotation, **stratified by their dominant impurity
category** (the category with the highest QuiltCleaner score). Each impurity type then appears in
the sample in proportion to its share of the impure pool.

The pool is all impure images (*Any impairment* ≥ 0.5). The 1,000 images annotated in this work are listed in
[`data/annotations/train.txt`](../data/annotations/train.txt).

**Output** → `outputs/annotation_sample.csv`, `outputs/annotation_images/`

In [ ]:
# --- setup: make the `requilt` package importable and load the shared config ---
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root (not needed after `pip install -e .`)

from requilt import load_config

cfg = load_config()      # configs/config.yaml
P = cfg.paths            # all paths, already resolved to absolute Paths

In [ ]:
import shutil

import pandas as pd

from requilt.data import ANY_IMPAIRMENT, add_dominant_impurity, stratified_allocation, stratified_sample

## 1 · Build the impure pool

In [ ]:
scores = pd.read_csv(P.output_dir / "quilt_impurity_scores.csv")

pool = scores[scores[ANY_IMPAIRMENT] >= cfg.impurity.threshold]
pool = add_dominant_impurity(pool)
print(f"impure images available for annotation: {len(pool):,}")

## 2 · Stratified sample of 1,000 images

In [ ]:
N = cfg.annotation.n_images
print("per-category allocation:")
print(stratified_allocation(pool["dominant_impurity"], N).to_string())

sample = stratified_sample(pool, N, seed=cfg.seed)
sample.to_csv(P.output_dir / "annotation_sample.csv", index=False)
print(f"\nsampled {len(sample)} images")

## 3 · Copy the images into a folder for upload to CVAT

In [ ]:
dst = P.output_dir / "annotation_images"
dst.mkdir(exist_ok=True)
for name in sample["Filename"]:
    shutil.copy2(P.quilt_images_dir / name, dst / name)
print(f"copied {len(sample)} images -> {dst}")

## 4 · Annotation protocol (CVAT)
Every image is annotated with bounding boxes for **four classes**:

| id | Class |
|:-:|---|
| 0 | `Persons/Photos` |
| 1 | `Desktop/Windows/SlideViewer` |
| 2 | `Text/Logo in Image` |
| 3 | `Pathology Tissue` |

Annotation focuses on delineating visible tissue regions while excluding surrounding elements
unrelated to tissue morphology. Software-interface elements along the edges or corners of the
tissue are kept out of the tissue boxes. Annotating the impurity classes as well allows the
detector to distinguish tissue from its surrounding context rather than recognizing tissue in
isolation.

Export the result with **Export → YOLO 1.1** and place it in `data/annotations/`.